# Day 2 - Supervised Learning: Regression

One notebook. Same student dataset. From raw data to trained regression models.

**Target:** Predict `final_exam_marks` from student attributes.

```
Part A: Machine Learning Foundations
   Step 1:  What is Machine Learning?
   Step 2:  Types of ML
   Step 3:  Regression vs Classification
   Step 4:  Features and Target
   Step 5:  Train-Test Split

Part B: Linear Regression
   Step 6:  Simple Linear Regression (one feature)
   Step 7:  Residuals — measuring error
   Step 8:  Multiple Linear Regression (all features)

Part C: Tree-Based Models
   Step 9:  Decision Tree Regression
   Step 10: Overfitting — the core danger
   Step 11: Random Forest Regression

Part D: Evaluation
   Step 12: MAE, MSE, RMSE — error metrics
   Step 13: R² Score — explained variance
   Step 14: Model Comparison
   Step 15: Actual vs Predicted & Residual Analysis

Part E: Automated ML with PyCaret
   Step 16: PyCaret setup and compare
   Step 17: Visualize and predict

Part F: Communication
   Step 18: What the models tell us about students
```

**Rule:** Understand the concept first, then use the tool.

---
## Step 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("Libraries ready.")

### Recreate and clean the Day 1 dataset

Same dataset, same seed, same cleaning steps. This ensures continuity across days.

In [ ]:
# ---- Recreate Day 1 dataset (same seed = same data) ----
np.random.seed(42)
n = 300

student_ids = np.arange(1001, 1001 + n)
first_names = ["Asha", "Ravi", "Meera", "John", "Fatima", "Chen", "Sara", "Vikram",
               "Nina", "Arjun", "Priya", "Amit", "Sneha", "Rahul", "Kavya", "Deepak",
               "Ananya", "Suresh", "Pooja", "Kiran", "Divya", "Manish", "Neha", "Rohan",
               "Swati", "Varun", "Ishita", "Gaurav", "Tanvi", "Siddharth"]
names = np.random.choice(first_names, n)

gender_clean = np.random.choice(["Male", "Female"], n, p=[0.55, 0.45])
gender_messy_map = {"Male": ["Male", "M", "m", "male", "MALE"], "Female": ["Female", "F", "f", "female", "FEMALE"]}
gender = np.array([np.random.choice(gender_messy_map[g]) for g in gender_clean])

dept_clean = np.random.choice(["CSE", "ECE", "ME", "CE", "EEE"], n, p=[0.3, 0.25, 0.2, 0.15, 0.1])
dept_messy_map = {"CSE": ["CSE", "cse", "Cse", "CSE ", " CSE"], "ECE": ["ECE", "ece", "Ece", "ECE "],
                  "ME": ["ME", "me", "Me", " ME"], "CE": ["CE", "ce", "Ce"], "EEE": ["EEE", "eee", "Eee"]}
department = np.array([np.random.choice(dept_messy_map[d]) for d in dept_clean])

semester = np.random.choice([1, 2, 3, 4, 5, 6], n)
attendance_pct = np.clip(np.random.normal(75, 15, n), 30, 100).round(1)
study_hours = np.clip(np.random.normal(15, 7, n), 2, 40).round(1)
assignment_score = np.clip(np.random.normal(65, 18, n), 10, 100).round(1)
lab_score = np.clip(np.random.normal(60, 15, n), 10, 100).round(1)
internal_marks = np.clip(np.random.normal(30, 8, n), 5, 50).round(1)

final_exam = np.clip(
    0.3 * attendance_pct + 0.8 * study_hours + 0.2 * assignment_score + np.random.normal(0, 8, n),
    10, 100
).round(1)

total_marks = (0.2 * assignment_score + 0.15 * lab_score + 0.25 * internal_marks + 0.4 * final_exam).round(1)
result = np.where(total_marks >= 75, "Distinction", np.where(total_marks >= 45, "Pass", "Fail"))

city_clean = np.random.choice(["Hyderabad", "Bangalore", "Chennai", "Mumbai", "Delhi"], n, p=[0.3, 0.25, 0.2, 0.15, 0.1])
city_messy_map = {"Hyderabad": ["Hyderabad", "hyderabad", " Hyderabad", "Hyderabad "],
                  "Bangalore": ["Bangalore", "bangalore", "Bangalore ", "BANGALORE"],
                  "Chennai": ["Chennai", "chennai", " Chennai"],
                  "Mumbai": ["Mumbai", "mumbai", "Mumbai "],
                  "Delhi": ["Delhi", "delhi", " Delhi"]}
city = np.array([np.random.choice(city_messy_map[c]) for c in city_clean])
extracurricular = np.random.choice(["Yes", "No"], n, p=[0.4, 0.6])

raw = pd.DataFrame({
    "student_id": student_ids, "name": names, "gender": gender, "department": department,
    "semester": semester, "attendance_pct": attendance_pct, "study_hours_per_week": study_hours,
    "assignment_score": assignment_score, "lab_score": lab_score, "internal_marks": internal_marks,
    "final_exam_marks": final_exam, "total_marks": total_marks, "result": result,
    "city": city, "extracurricular": extracurricular,
})

# Inject data quality problems (same as Day 1)
miss_idx = np.random.choice(n, 8, replace=False)
raw.loc[miss_idx[0:3], "attendance_pct"] = np.nan
raw.loc[miss_idx[3:5], "study_hours_per_week"] = np.nan
raw.loc[miss_idx[5:7], "assignment_score"] = np.nan
raw.loc[miss_idx[7], "extracurricular"] = np.nan
extra_miss = np.random.choice(n, 2, replace=False)
raw.loc[extra_miss, "extracurricular"] = np.nan
raw.loc[10, "attendance_pct"] = 105.0
raw.loc[55, "attendance_pct"] = 110.5
raw.loc[20, "study_hours_per_week"] = -3.0
raw.loc[88, "study_hours_per_week"] = -5.0
raw.loc[150, "study_hours_per_week"] = 50.0
dup_rows = raw.iloc[[5, 42, 99]].copy()
raw = pd.concat([raw, dup_rows], ignore_index=True)

# ---- Apply Day 1 cleaning ----
df = raw.copy()
df["gender"] = df["gender"].str.strip().str.lower().replace({"m": "male", "f": "female"}).str.title()
df["department"] = df["department"].str.strip().str.upper()
df["city"] = df["city"].str.strip().str.title()
df = df.drop_duplicates()
for col in ["attendance_pct", "study_hours_per_week", "assignment_score"]:
    df[col] = df[col].fillna(df[col].median())
df["extracurricular"] = df["extracurricular"].fillna("Unknown")
df.loc[df["attendance_pct"] > 100, "attendance_pct"] = 100.0
study_median = df.loc[df["study_hours_per_week"] >= 0, "study_hours_per_week"].median()
df.loc[df["study_hours_per_week"] < 0, "study_hours_per_week"] = study_median

print(f"Cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing values: {df.isna().sum().sum()}")
df.head()

---
# Part A: Machine Learning Foundations

---
## Step 1: What is Machine Learning?

Yesterday we explored data, found patterns, and wrote observations. But we did all the thinking ourselves. What if we could teach a computer to find those patterns automatically — and use them to make predictions on new data it has never seen?

That is Machine Learning.

### Traditional programming vs Machine Learning

| Approach | Input | Output |
|---|---|---|
| **Traditional** | Data + Rules (written by human) | Answers |
| **Machine Learning** | Data + Answers (examples) | Rules (learned by algorithm) |

Traditional programming:
```
IF attendance > 75 AND study_hours > 20 THEN predict high marks
```
You write every rule by hand. Works for simple cases, but breaks with complex patterns.

Machine Learning:
```
Give the algorithm many (attendance, study_hours, marks) examples.
It learns: "students with attendance ~80 and study_hours ~25 tend to score ~60."
```
The algorithm discovers the rules from data. The more good examples, the better the rules.

### When to use ML?

| Use ML when | Don't use ML when |
|---|---|
| Patterns are complex and hard to write as rules | Rules are simple and well-known |
| You have enough historical data with outcomes | You have very little data |
| The problem repeats and needs automation | It's a one-time analysis |
| Accuracy matters and can be measured | The problem needs human judgment only |

---
## Step 2: Types of Machine Learning

| Type | Has labeled answers? | Goal | Example |
|---|---|---|---|
| **Supervised** | Yes — we know the right answers | Learn to predict the answer | Predict marks from attendance |
| **Unsupervised** | No — no right answers given | Find hidden structure | Group similar students |
| **Semi-supervised** | Some labeled, most unlabeled | Learn from both | Classify with few labeled examples |
| **Reinforcement** | Reward/penalty signals | Learn by trial and error | Game-playing AI, robotics |

### Supervised Learning is the workhorse

Most real-world ML starts with supervised learning because we usually have historical data with known outcomes.

**"Supervised"** means the algorithm learns with a teacher — the teacher is the labeled data.

We give it input-output pairs:
```
(attendance=80, study_hours=20) --> final_exam=55
(attendance=90, study_hours=30) --> final_exam=72
```

The algorithm finds the pattern connecting inputs to outputs. Then we give it new inputs (a student it hasn't seen) and ask: what output do you predict?

---
## Step 3: Regression vs Classification

Supervised learning has two sub-types, based on what you're predicting:

| Task | What you predict | Target type | Example |
|---|---|---|---|
| **Regression** | A number | Continuous | Predict marks: 45.2, 67.8, 82.1 |
| **Classification** | A category | Discrete | Predict result: Pass / Fail |

How to tell which one you need:
- "How much?" or "How many?" --> Regression
- "Which category?" or "Yes or No?" --> Classification

| Question | Task |
|---|---|
| What will this student score on the final exam? | Regression |
| Will this student pass or fail? | Classification |
| How many hours will the project take? | Regression |
| Is this email spam or not spam? | Classification |

**Today: Regression** — predicting `final_exam_marks` (a number).

**Day 3: Classification** — predicting `result` (Pass / Fail).

---
## Step 4: Features and Target

Every supervised learning problem has two parts:

| Term | Meaning | Also called |
|---|---|---|
| **Features (X)** | Input columns used to make predictions | Independent variables, predictors |
| **Target (y)** | The column we want to predict | Dependent variable, label |

```
Features (X)                                          Target (y)
attendance, study_hours, assignment, lab, internal --> final_exam_marks
```

### Choosing features wisely

Not every column should be a feature:

| Column | Use as feature? | Why |
|---|---|---|
| attendance_pct | Yes | Likely affects exam performance |
| study_hours_per_week | Yes | Directly related to learning |
| assignment_score | Yes | Measures ongoing performance |
| lab_score | Yes | Another performance indicator |
| internal_marks | Yes | Assessment before final exam |
| student_id | No | Just an identifier, no predictive value |
| name | No | Personal label, not a pattern |
| total_marks | No | Calculated FROM final_exam_marks — data leakage |
| result | No | Derived FROM total_marks — data leakage |

**Data leakage** = using information that wouldn't be available at prediction time. If `total_marks` includes `final_exam_marks`, using it to predict `final_exam_marks` is cheating — the answer is already baked into the input.

In [ ]:
# Define features and target
feature_cols = ["attendance_pct", "study_hours_per_week", "assignment_score",
                "lab_score", "internal_marks"]
target_col = "final_exam_marks"

X = df[feature_cols]
y = df[target_col]

print(f"Features (X): {X.shape[1]} columns, {X.shape[0]} rows")
print(f"  Columns: {list(X.columns)}")
print(f"\nTarget (y): {target_col}")
print(f"  Range: {y.min()} to {y.max()}")
print(f"  Mean:  {y.mean():.1f}")

### Understand the target before modeling

Before building any model, understand what you're predicting. What does the distribution of `final_exam_marks` look like? Is it symmetric? Skewed? Are there outliers?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution
axes[0].hist(y, bins=20, color="#2F80ED", edgecolor="white", alpha=0.8)
axes[0].axvline(y.mean(), color="red", linestyle="--", label=f"Mean={y.mean():.1f}")
axes[0].axvline(y.median(), color="green", linestyle="--", label=f"Median={y.median():.1f}")
axes[0].set_title(f"Distribution of {target_col}")
axes[0].set_xlabel(target_col)
axes[0].set_ylabel("Count")
axes[0].legend()

# Box plot
axes[1].boxplot(y, vert=True)
axes[1].set_title(f"Box Plot of {target_col}")
axes[1].set_ylabel(target_col)

plt.tight_layout()
plt.show()

print(f"Skewness: {y.skew():.3f}")
print(f"The target is roughly symmetric — good for linear regression.")

### Which features correlate with the target?

From Day 1, we know correlation measures how strongly two variables move together. Let's check which features are most correlated with `final_exam_marks`. Stronger correlation = more useful for prediction.

In [ ]:
# Correlation of each feature with the target
correlations = X.corrwith(y).sort_values(ascending=False)

print("Correlation with final_exam_marks:")
for feat, corr in correlations.items():
    strength = "strong" if abs(corr) >= 0.7 else "moderate" if abs(corr) >= 0.3 else "weak"
    print(f"  {feat:25s}  r = {corr:+.3f}  ({strength})")

In [ ]:
# Scatter plots: each feature vs target
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, col in zip(axes, feature_cols):
    ax.scatter(df[col], y, alpha=0.3, s=15, color="#2F80ED")
    ax.set_xlabel(col)
    ax.set_ylabel(target_col if col == feature_cols[0] else "")
    ax.set_title(f"r = {df[col].corr(y):.3f}")
plt.tight_layout()
plt.show()

print("Upward trend = positive correlation. Tighter cloud = stronger relationship.")

---
## Step 5: Train-Test Split

### Why can't we train and test on the same data?

Imagine a student who memorizes last year's exam paper word-for-word. They score 100% on that paper. But give them a new paper — they fail. They memorized, they didn't learn.

ML models can do the same thing. If we train a model on all our data and then test it on the same data, we're measuring memorization, not learning. We need to test on data the model has **never seen**.

### The solution: split the data

```
Full Data (300 rows)
  |--- Training Set (80% = 240 rows) --> Model learns from this
  |--- Test Set (20% = 60 rows)      --> Model is evaluated on this (never seen)
```

| Set | Purpose | Model sees it during training? |
|---|---|---|
| Training | Learn patterns | Yes |
| Test | Evaluate on unseen data | No — held back |

The test set simulates "the real world" — future students the model hasn't encountered.

### Important rules

1. **Split before any modeling** — never let test data influence training
2. **Use random_state for reproducibility** — same split every time you run
3. **Typical split:** 80/20 or 70/30

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:     {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nTraining target mean: {y_train.mean():.1f}")
print(f"Test target mean:     {y_test.mean():.1f}")
print(f"\nSimilar means = good split (no accidental bias).")

---
# Part B: Linear Regression

---
## Step 6: Simple Linear Regression

### The idea

The simplest ML model: fit a straight line through the data.

```
y = m * x + b
final_exam_marks = slope * study_hours + intercept
```

- **Slope (m):** How much `y` changes when `x` increases by 1 unit
- **Intercept (b):** The predicted `y` when `x` is 0

### How does the algorithm find the best line?

It tries many possible lines and picks the one that **minimizes the total error** — the sum of squared distances between each point and the line. This is called **Ordinary Least Squares (OLS)**.

Think of it as: the line that is as close as possible to all points at once.

### Why start with one feature?

Starting with one feature lets us visualize the line on a 2D plot. Once the concept clicks, we'll add more features.

In [ ]:
# Simple regression: study_hours_per_week --> final_exam_marks
X_simple_train = X_train[["study_hours_per_week"]]
X_simple_test = X_test[["study_hours_per_week"]]

model_simple = LinearRegression()
model_simple.fit(X_simple_train, y_train)

slope = model_simple.coef_[0]
intercept = model_simple.intercept_

print(f"Slope (m):     {slope:.4f}")
print(f"Intercept (b): {intercept:.4f}")
print(f"\nFormula: final_exam = {slope:.2f} * study_hours + {intercept:.2f}")
print(f"\nInterpretation:")
print(f"  - Each extra hour of weekly study adds ~{slope:.2f} marks to the final exam.")
print(f"  - A student with 0 study hours would be predicted to score {intercept:.1f}.")

In [ ]:
# Visualize the regression line
plt.figure(figsize=(9, 5))
plt.scatter(X_simple_test, y_test, alpha=0.5, label="Actual (test data)", color="#2F80ED")

# Draw regression line
x_line = np.linspace(X["study_hours_per_week"].min(), X["study_hours_per_week"].max(), 100).reshape(-1, 1)
y_line = model_simple.predict(x_line)
plt.plot(x_line, y_line, color="red", linewidth=2, label=f"y = {slope:.2f}x + {intercept:.1f}")

plt.xlabel("Study Hours per Week")
plt.ylabel("Final Exam Marks")
plt.title("Simple Linear Regression: Study Hours vs Final Exam")
plt.legend()
plt.show()

print("The line captures the general upward trend.")
print("But many points are far from the line — one feature isn't enough.")

In [ ]:
# Predict and compare
y_pred_simple = model_simple.predict(X_simple_test)

comparison_simple = pd.DataFrame({
    "study_hours": X_simple_test["study_hours_per_week"].values,
    "actual": y_test.values,
    "predicted": y_pred_simple.round(1),
    "error": (y_test.values - y_pred_simple).round(1),
})
comparison_simple.head(10)

---
## Step 7: Residuals — Measuring Error

The difference between what a model predicts and what actually happened is called a **residual**.

```
Residual = Actual - Predicted
```

| Residual | Meaning |
|---|---|
| Positive | Model predicted too low (actual was higher) |
| Negative | Model predicted too high (actual was lower) |
| Zero | Perfect prediction |

### What good residuals look like

A well-fitting model should have residuals that:
1. **Are centered around zero** — the model isn't systematically too high or too low
2. **Show no pattern** — errors are random, not structured
3. **Are small** — predictions are close to reality

If residuals show a pattern (e.g., errors increase with higher predictions), the model is missing something.

In [ ]:
# Residual plot for simple regression
residuals_simple = y_test.values - y_pred_simple

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Residuals vs predicted
axes[0].scatter(y_pred_simple, residuals_simple, alpha=0.5, color="#F2994A")
axes[0].axhline(y=0, color="red", linestyle="--", linewidth=1)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Residual (Actual - Predicted)")
axes[0].set_title("Residuals vs Predicted (Simple)")

# Residual distribution
axes[1].hist(residuals_simple, bins=15, color="#F2994A", edgecolor="white", alpha=0.8)
axes[1].axvline(x=0, color="red", linestyle="--", linewidth=1)
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution (Simple)")

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals_simple.mean():.2f} (should be near 0)")
print(f"Std of residuals: {residuals_simple.std():.2f} (smaller = better)")
print(f"\nResiduals are centered near 0 but wide — the model has high variance in errors.")

---
## Step 8: Multiple Linear Regression

### Why one feature isn't enough

A student's exam score depends on more than just study hours. Attendance, assignments, labs, and internal marks all play a role. Simple linear regression ignores all of that.

Multiple linear regression uses **all features together**:

```
final_exam = w1*attendance + w2*study_hours + w3*assignment + w4*lab + w5*internal + b
```

Each feature gets its own **weight (coefficient)** showing how much it contributes to the prediction.

### What the coefficients mean

- **Positive coefficient:** Increasing this feature increases the predicted marks
- **Negative coefficient:** Increasing this feature decreases the predicted marks
- **Larger absolute value:** Stronger influence on prediction

**Caution:** Coefficients are only directly comparable when features are on the same scale. Our features have different ranges (attendance 0-100, internal_marks 0-50), so compare with care.

In [ ]:
model_multi = LinearRegression()
model_multi.fit(X_train, y_train)

# Coefficients
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model_multi.coef_.round(4),
}).sort_values("coefficient", key=abs, ascending=False)

print(f"Intercept: {model_multi.intercept_:.4f}")
print(f"\nCoefficients (sorted by absolute impact):")
for _, row in coef_df.iterrows():
    direction = "increases" if row["coefficient"] > 0 else "decreases"
    print(f"  {row['feature']:25s}  {row['coefficient']:+.4f}  "
          f"(+1 unit {direction} prediction by {abs(row['coefficient']):.2f})")

In [ ]:
# Visualize coefficients
plt.figure(figsize=(8, 4))
colors = ["#27AE60" if c > 0 else "#E74C3C" for c in coef_df["coefficient"]]
plt.barh(coef_df["feature"], coef_df["coefficient"], color=colors)
plt.xlabel("Coefficient (impact on final_exam_marks)")
plt.title("Feature Coefficients: Multiple Linear Regression")
plt.axvline(x=0, color="gray", linestyle="--", linewidth=0.8)
plt.show()

print("Green = positive impact on marks. Red = negative impact.")

In [ ]:
# Predict on test set
y_pred_multi = model_multi.predict(X_test)

comparison_multi = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred_multi.round(1),
    "error": (y_test.values - y_pred_multi).round(1),
})

print("Predictions (first 10):")
comparison_multi.head(10)

### Simple vs Multiple: Quick comparison

In [ ]:
r2_simple = r2_score(y_test, y_pred_simple)
r2_multi = r2_score(y_test, y_pred_multi)

print(f"Simple Linear (1 feature):   R² = {r2_simple:.4f}")
print(f"Multiple Linear (5 features): R² = {r2_multi:.4f}")
print(f"\nImprovement: {r2_multi - r2_simple:.4f}")
print(f"\nMore features = more information = better predictions.")

---
# Part C: Tree-Based Models

---
## Step 9: Decision Tree Regression

### The idea

Linear regression draws a straight line (or plane). But what if the relationship isn't linear? What if students with very low attendance score differently than what a line predicts?

A decision tree takes a different approach: it **splits the data into regions** using if-else rules, then predicts the average of each region.

```
Is study_hours > 15?
  Yes --> Is attendance > 80?
            Yes --> Predict 62
            No  --> Predict 51
  No  --> Is assignment_score > 60?
            Yes --> Predict 45
            No  --> Predict 35
```

### How the tree decides where to split

At each step, the tree tries every possible split point on every feature and picks the one that **reduces prediction error the most**. It keeps splitting until it reaches a stopping rule (like max depth).

### Pros and cons

| Pros | Cons |
|---|---|
| Captures non-linear patterns | Can overfit easily |
| Easy to interpret (if-else rules) | Unstable — small data changes can change the tree |
| Handles different feature scales naturally | Single tree can be high-variance |

In [ ]:
model_tree = DecisionTreeRegressor(max_depth=4, random_state=42)
model_tree.fit(X_train, y_train)

y_pred_tree = model_tree.predict(X_test)

print(f"Tree depth: {model_tree.get_depth()}")
print(f"Number of leaves (prediction regions): {model_tree.get_n_leaves()}")
print(f"R² on test set: {r2_score(y_test, y_pred_tree):.4f}")

In [ ]:
# Feature importance from the tree
tree_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model_tree.feature_importances_.round(4),
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 4))
plt.barh(tree_importance["feature"], tree_importance["importance"], color="#F2994A")
plt.xlabel("Importance (fraction of variance explained)")
plt.title("Feature Importance: Decision Tree")
plt.show()

print("Feature importance shows which features the tree relied on most for splitting.")
print(tree_importance.to_string(index=False))

---
## Step 10: Overfitting — The Core Danger

### What is overfitting?

A model that **memorizes** the training data instead of learning the underlying pattern. It performs brilliantly on training data but poorly on new data.

| | Underfitting | Good Fit | Overfitting |
|---|---|---|---|
| **Training performance** | Poor | Good | Excellent |
| **Test performance** | Poor | Good | Poor |
| **Problem** | Too simple | Just right | Too complex |
| **Analogy** | Student who didn't study | Student who understood concepts | Student who memorized answers |

### How to detect overfitting

Compare training score vs test score. A large gap means overfitting.

### Decision trees are prone to overfitting

An unrestricted tree will keep splitting until every training sample has its own leaf. Perfect training score, terrible test score. We control this with `max_depth`.

In [ ]:
# Demonstrate overfitting: deep tree vs shallow tree
print(f"{'Depth':<8} {'Train R²':<12} {'Test R²':<12} {'Gap':<12} {'Verdict'}")
print("-" * 60)

for depth in [2, 3, 4, 5, 8, 15, None]:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_r2 = r2_score(y_train, dt.predict(X_train))
    test_r2 = r2_score(y_test, dt.predict(X_test))
    gap = train_r2 - test_r2
    depth_str = str(depth) if depth else "None"
    verdict = "Underfitting" if test_r2 < 0.3 else "Overfitting" if gap > 0.15 else "Good"
    print(f"{depth_str:<8} {train_r2:<12.4f} {test_r2:<12.4f} {gap:<12.4f} {verdict}")

print(f"\nAs depth increases:")
print(f"  - Train R² approaches 1.0 (memorization)")
print(f"  - Test R² improves then drops (overfitting)")
print(f"  - The gap widens = classic overfitting signal")

---
## Step 11: Random Forest Regression

### The problem with a single tree

A single decision tree is unstable — change a few training rows and you might get a completely different tree. It's also prone to overfitting.

### The solution: build many trees

A Random Forest builds **many decision trees**, each trained on a random subset of the data and features. Then it **averages** all their predictions.

```
Tree 1 (random subset): predicts 55
Tree 2 (random subset): predicts 60
Tree 3 (random subset): predicts 52
...
Tree 100: predicts 58

Random Forest prediction: average = 56.25
```

### Why averaging helps

Individual trees may overfit in different ways. But their errors tend to cancel out when averaged. This is called **ensemble learning** — combining weak models to create a strong one.

| Single Tree | Random Forest |
|---|---|
| One opinion | 100 opinions averaged |
| Unstable | Stable |
| Prone to overfitting | Resistant to overfitting |
| Fast to train | Slower (many trees) |

### Key parameters

| Parameter | Meaning | Typical value |
|---|---|---|
| `n_estimators` | Number of trees | 100-500 |
| `max_depth` | Maximum depth per tree | 4-10 |
| `random_state` | Reproducibility seed | 42 |

In [ ]:
model_rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

train_r2_rf = r2_score(y_train, model_rf.predict(X_train))
test_r2_rf = r2_score(y_test, y_pred_rf)

print(f"Number of trees: {model_rf.n_estimators}")
print(f"Max depth per tree: {model_rf.max_depth}")
print(f"\nTrain R²: {train_r2_rf:.4f}")
print(f"Test R²:  {test_r2_rf:.4f}")
print(f"Gap:      {train_r2_rf - test_r2_rf:.4f}  (small gap = healthy model)")

In [ ]:
# Feature importance from Random Forest
rf_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model_rf.feature_importances_.round(4),
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 4))
plt.barh(rf_importance["feature"], rf_importance["importance"], color="#9B51E0")
plt.xlabel("Importance")
plt.title("Feature Importance: Random Forest")
plt.show()

print("Random Forest importance is more stable than a single tree because it averages across 100 trees.")
print(rf_importance.to_string(index=False))

---
# Part D: Evaluation

We've built four models. How do we decide which one is best? We need **metrics** — numbers that measure prediction quality.

---
## Step 12: MAE, MSE, RMSE — Error Metrics

### The intuition: build up from simple to complete

Just like Day 1 built from Range to Variance to Std Dev, we'll build error metrics step by step.

**The basic question:** How far are predictions from reality?

### Step 12a: MAE — Mean Absolute Error

Take each error, make it positive (absolute value), then average.

```
MAE = mean( |actual - predicted| )
```

**Interpretation:** "On average, our predictions are off by MAE marks."

**Units:** Same as the target (marks). Easy to explain.

### Step 12b: MSE — Mean Squared Error

Take each error, square it, then average.

```
MSE = mean( (actual - predicted)² )
```

**Why square?** Squaring penalizes large errors more than small ones. An error of 20 contributes 400, while an error of 5 contributes only 25. This is useful when big mistakes are much worse than small ones.

**Problem:** Units are squared (marks²) — hard to interpret directly.

### Step 12c: RMSE — Root Mean Squared Error

Take the square root of MSE to get back to original units.

```
RMSE = sqrt(MSE)
```

**Interpretation:** Like MAE but gives more weight to large errors.

**RMSE >= MAE always.** If RMSE is much larger than MAE, the model has some very large errors.

### Summary

| Metric | Formula | Units | Penalizes large errors? |
|---|---|---|---|
| MAE | mean(\|error\|) | Marks | Equally |
| MSE | mean(error²) | Marks² | Heavily |
| RMSE | sqrt(MSE) | Marks | Heavily |

In [ ]:
# Manual calculation to build intuition
actual = np.array([50, 60, 70, 40, 80])
predicted = np.array([48, 63, 68, 55, 78])
errors = actual - predicted

demo = pd.DataFrame({
    "actual": actual,
    "predicted": predicted,
    "error": errors,
    "|error|": np.abs(errors),
    "error_squared": errors ** 2,
})

print(demo.to_string(index=False))
print(f"\n--- Calculations ---")
print(f"MAE  = mean of |error|  = {np.abs(errors).mean():.1f} marks")
print(f"MSE  = mean of error²   = {(errors**2).mean():.1f} marks²")
print(f"RMSE = sqrt(MSE)        = {np.sqrt((errors**2).mean()):.1f} marks")
print(f"\nNotice: RMSE ({np.sqrt((errors**2).mean()):.1f}) > MAE ({np.abs(errors).mean():.1f})")
print(f"because the large error (student 4: off by 15) gets penalized more by squaring.")

---
## Step 13: R² Score — Explained Variance

MAE and RMSE tell you the **size** of errors, but they don't tell you if the errors are good or bad relative to the problem. An RMSE of 5 is great for salary prediction but terrible for predicting coin flips.

R² answers: **"What fraction of the target's variance does the model explain?"**

### The baseline: predicting the mean

The dumbest possible model: always predict the average of `y` regardless of input.

```
Baseline: predict mean(final_exam_marks) for every student = ~48
```

R² measures how much better your model is compared to this dumb baseline.

```
R² = 1 - (model errors / baseline errors)
```

| R² Value | Meaning |
|---|---|
| 1.0 | Perfect — model explains all variance |
| 0.8 | Good — model explains 80% of variance |
| 0.5 | Moderate — model explains half the variance |
| 0.0 | Useless — model is no better than predicting the mean |
| < 0 | Harmful — model is worse than predicting the mean |

In [ ]:
# Manual R² calculation
y_mean = actual.mean()

ss_res = ((actual - predicted) ** 2).sum()    # model errors
ss_tot = ((actual - y_mean) ** 2).sum()        # baseline errors (predicting mean)
r2_manual = 1 - ss_res / ss_tot

print(f"Mean of actual values: {y_mean}")
print(f"\nBaseline (predict mean for everyone):")
print(f"  Total squared error: {ss_tot:.0f}")
print(f"\nOur model:")
print(f"  Total squared error: {ss_res:.0f}")
print(f"\nR² = 1 - {ss_res:.0f}/{ss_tot:.0f} = {r2_manual:.4f}")
print(f"\nMeaning: Our model explains {r2_manual*100:.1f}% of the variance in exam marks.")
print(f"The remaining {(1-r2_manual)*100:.1f}% is due to factors we didn't capture.")

---
## Step 14: Model Comparison

Now let's compare all four models using all metrics.

In [ ]:
def evaluate(name, y_true, y_pred):
    return {
        "Model": name,
        "MAE": round(mean_absolute_error(y_true, y_pred), 2),
        "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        "R²": round(r2_score(y_true, y_pred), 4),
    }

results = pd.DataFrame([
    evaluate("Simple Linear (1 feature)", y_test, y_pred_simple),
    evaluate("Multiple Linear (5 features)", y_test, y_pred_multi),
    evaluate("Decision Tree (depth=4)", y_test, y_pred_tree),
    evaluate("Random Forest (100 trees)", y_test, y_pred_rf),
])

results

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#E74C3C", "#2F80ED", "#F2994A", "#9B51E0"]

# R² comparison
axes[0].bar(results["Model"], results["R²"], color=colors)
axes[0].set_ylabel("R² Score")
axes[0].set_title("Model Comparison: R² (higher = better)")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=25)
for i, v in enumerate(results["R²"]):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold", fontsize=10)

# RMSE comparison
axes[1].bar(results["Model"], results["RMSE"], color=colors)
axes[1].set_ylabel("RMSE (marks)")
axes[1].set_title("Model Comparison: RMSE (lower = better)")
axes[1].tick_params(axis='x', rotation=25)
for i, v in enumerate(results["RMSE"]):
    axes[1].text(i, v + 0.2, f"{v:.1f}", ha="center", fontweight="bold", fontsize=10)

plt.tight_layout()
plt.show()

### Reading the comparison

| What to look for | Meaning |
|---|---|
| Highest R² | Explains the most variance |
| Lowest RMSE | Smallest prediction errors |
| Lowest MAE | Smallest average error |
| Simple > Multiple? | More features helped |
| Tree vs Forest? | Ensemble usually wins |

---
## Step 15: Actual vs Predicted & Residual Analysis

### Actual vs Predicted plot

Plot actual values (x-axis) against predicted values (y-axis). If the model were perfect, all points would fall on the diagonal line (y = x). Points closer to the line = better predictions.

In [ ]:
# Actual vs Predicted for all models
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
predictions = {
    "Simple Linear": y_pred_simple,
    "Multiple Linear": y_pred_multi,
    "Decision Tree": y_pred_tree,
    "Random Forest": y_pred_rf,
}

for ax, (name, y_pred), color in zip(axes, predictions.items(), colors):
    ax.scatter(y_test, y_pred, alpha=0.5, color=color, s=30)
    min_val = min(y_test.min(), y_pred.min()) - 5
    max_val = max(y_test.max(), y_pred.max()) + 5
    ax.plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1, alpha=0.5)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    r2 = r2_score(y_test, y_pred)
    ax.set_title(f"{name}\nR²={r2:.3f}")
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)

plt.tight_layout()
plt.show()

print("Closer to the diagonal = better model.")
print("Wider scatter = more prediction error.")

### Residual analysis for the best model

Good residuals should be:
1. Centered around zero
2. Randomly scattered (no pattern)
3. Roughly normally distributed

In [ ]:
# Residual analysis for the best model (by R²)
best_idx = results["R²"].idxmax()
best_name = results.loc[best_idx, "Model"]
best_preds = list(predictions.values())[best_idx]
best_residuals = y_test.values - best_preds

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 1. Residuals vs Predicted
axes[0].scatter(best_preds, best_residuals, alpha=0.5, color="#27AE60")
axes[0].axhline(y=0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs Predicted")

# 2. Residual distribution
axes[1].hist(best_residuals, bins=15, color="#27AE60", edgecolor="white", alpha=0.8)
axes[1].axvline(x=0, color="red", linestyle="--")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")

# 3. Q-Q style: sorted residuals
sorted_res = np.sort(best_residuals)
axes[2].scatter(range(len(sorted_res)), sorted_res, alpha=0.5, color="#27AE60", s=15)
axes[2].axhline(y=0, color="red", linestyle="--")
axes[2].set_xlabel("Index (sorted)")
axes[2].set_ylabel("Residual")
axes[2].set_title("Sorted Residuals")

plt.suptitle(f"Residual Analysis: {best_name}", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Mean residual:  {best_residuals.mean():.2f} (should be ~0)")
print(f"Std residual:   {best_residuals.std():.2f}")
print(f"Max overpredict: {best_residuals.min():.1f}")
print(f"Max underpredict: {best_residuals.max():.1f}")

### Which metric should you report?

| Audience | Report |
|---|---|
| **Technical team** | R², RMSE, MAE, residual plots |
| **Business stakeholder** | MAE ("predictions are off by ~X marks on average") |
| **Quick comparison** | R² (higher = better, 0 to 1 scale) |

Always report at least R² and one error metric (RMSE or MAE).

---
# Part E: Automated ML with PyCaret

---
## Step 16: PyCaret — Compare Many Models Instantly

So far we manually created 4 models. What if we could compare 15+ algorithms in one line?

**PyCaret** is a low-code ML library that automates the workflow.

| Scikit-learn | PyCaret |
|---|---|
| Manual: create each model, train, evaluate | Automated: one line compares all models |
| Full control, more code | Quick prototyping, less code |
| You learn how it works | You get fast results |

**Both are valuable.** Scikit-learn teaches the mechanics (what we just did). PyCaret helps you move fast in practice.

### PyCaret workflow

```
1. setup()          --> configure experiment
2. compare_models() --> train and rank all algorithms
3. create_model()   --> train a specific model
4. plot_model()     --> visualize performance
5. predict_model()  --> predict on new data
```

In [ ]:
# Uncomment and run if PyCaret is not installed:
# !pip install pycaret

In [ ]:
from pycaret.regression import *

# Prepare a clean DataFrame for PyCaret (only features + target)
df_pycaret = df[feature_cols + [target_col]].copy()

reg_setup = setup(
    data=df_pycaret,
    target=target_col,
    session_id=42,
    verbose=False,
)

print("PyCaret setup complete.")

In [ ]:
# Compare all regression models — one line!
best = compare_models()

In [ ]:
# What model won?
print(f"Best model: {type(best).__name__}")
print(best)

---
## Step 17: Visualize and Predict with PyCaret

In [ ]:
# Create a Random Forest model in PyCaret for comparison
rf_pycaret = create_model("rf")

In [ ]:
# Residual plot
plot_model(rf_pycaret, plot="residuals")

In [ ]:
# Actual vs Predicted
plot_model(rf_pycaret, plot="error")

In [ ]:
# Feature importance
plot_model(rf_pycaret, plot="feature")

In [ ]:
# Predict on held-out test set
pycaret_predictions = predict_model(rf_pycaret)
pycaret_predictions.head(10)

In [ ]:
# Predict for completely new students
new_students = pd.DataFrame({
    "attendance_pct": [90, 50, 75],
    "study_hours_per_week": [25, 8, 15],
    "assignment_score": [80, 40, 65],
    "lab_score": [75, 45, 60],
    "internal_marks": [35, 15, 28],
})

print("New students:")
display(new_students)

new_preds = predict_model(rf_pycaret, data=new_students)
print("\nPredictions:")
new_preds

---
# Part F: Communication

---
## Step 18: What the Models Tell Us About Students

In [ ]:
# Final comparison table
print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(results.to_string(index=False))
print("=" * 60)

In [ ]:
# What did we learn about student performance?
best_model_r2 = results.loc[results["R²"].idxmax()]

report = f"""DAY 2 REGRESSION REPORT: PREDICTING FINAL EXAM MARKS
{'='*55}

TARGET: final_exam_marks
FEATURES: {', '.join(feature_cols)}
DATASET: {len(df)} students (Train: {len(X_train)}, Test: {len(X_test)})

BEST MODEL: {best_model_r2['Model']}
  R²:   {best_model_r2['R²']:.4f} (explains {best_model_r2['R²']*100:.1f}% of variance)
  RMSE: {best_model_r2['RMSE']:.2f} marks
  MAE:  {best_model_r2['MAE']:.2f} marks

KEY FINDINGS:
1. Using all 5 features is much better than study hours alone.
2. Feature importance (Random Forest):
{chr(10).join(f'   - {row["feature"]:25s} {row["importance"]:.3f}' for _, row in rf_importance.iterrows())}
3. Tree-based models capture non-linear patterns that linear
   regression misses.
4. Random Forest reduces overfitting compared to a single tree.

PRACTICAL MEANING:
- The model can predict a student's final exam marks within
  ~{best_model_r2['MAE']:.0f} marks on average.
- Students with higher attendance, more study hours, and better
  assignment scores tend to score higher on the final exam.
- This model could help identify at-risk students early.

CAUTION:
- Synthetic dataset — real data may show different patterns.
- Correlation is not causation — more study hours is associated
  with better scores, but other factors may be involved.
- Model should be validated on truly unseen data before deployment.
"""

print(report)

---
## Recap

| Step | What We Learned | Key Tool |
|---|---|---|
| 1. What is ML? | Algorithms learn patterns from data | Concept |
| 2. Types of ML | Supervised, Unsupervised, Semi-supervised, RL | Concept |
| 3. Reg vs Class | Regression predicts numbers, Classification predicts categories | Concept |
| 4. Features & Target | X = inputs, y = what we predict; avoid data leakage | `df[features]`, `df[target]` |
| 5. Train-Test Split | Test on unseen data to measure real performance | `train_test_split()` |
| 6. Simple Linear | One feature, one line, slope and intercept | `LinearRegression()` |
| 7. Residuals | Error = Actual - Predicted; should be random and centered at 0 | Scatter/histogram |
| 8. Multiple Linear | All features together; coefficients show each feature's impact | `LinearRegression()` |
| 9. Decision Tree | Splits data into regions with if-else rules | `DecisionTreeRegressor()` |
| 10. Overfitting | Memorizing training data; detect via train-test gap | Concept |
| 11. Random Forest | Many trees averaged; reduces overfitting | `RandomForestRegressor()` |
| 12. MAE/MSE/RMSE | Error metrics in original units; RMSE penalizes large errors | `mean_absolute_error()` |
| 13. R² Score | Fraction of variance explained (0 to 1) | `r2_score()` |
| 14. Model Comparison | Compare all models on the same test set | DataFrame |
| 15. Diagnostics | Actual vs Predicted plot; residual analysis | matplotlib |
| 16-17. PyCaret | Automated model comparison and visualization | `setup()`, `compare_models()` |
| 18. Communication | Report findings with evidence and caution | Plain English |

### The scikit-learn pattern (memorize this)

```python
# 1. Prepare
X = df[feature_columns]
y = df[target_column]

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 3. Create and train
model = SomeRegressor()
model.fit(X_train, y_train)

# 4. Predict and evaluate
y_pred = model.predict(X_test)
print(r2_score(y_test, y_pred))
```

This same 6-line pattern works for ANY scikit-learn model.

---

**Next: Day 3 — Classification (predicting Pass/Fail instead of marks)**

Same dataset. Same features. Different target: `result` (Pass / Fail). Different algorithms. Different metrics.